# P12 — Forehead Creases PAD: InceptionV3 Binary Classification
**Project:** Forehead Creases Based Presentation Attack Detection  
**Mentor:** Geetanjali Sharma (d19062@students.iitmandi.ac.in)  
**Task:** Binary Classification — Real vs Attack (Printed)  
**Model:** Pretrained InceptionV3 (ImageNet)  

---
Two pipelines trained and compared:
- **Pipeline A** — InceptionV3 **without** augmentation  
- **Pipeline B** — InceptionV3 **with** augmentation (HFlip + Rotation ±20°)  

All results (plots, metrics, models) are saved to `output_inceptionv3/`

## 1. Install & Import Dependencies

In [1]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "pexpect"])

0

In [2]:
!pip install torch torchvision scikit-learn matplotlib seaborn tqdm --quiet

In [3]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "torch", "torchvision"])

  Using cached torch-2.11.0-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached torchvision-0.26.0-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached triton-3.6.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (1.7 kB)
Using cached torch-2.11.0-cp310-cp310-manylinux_2_28_x86_64.whl (530.6 MB)
Using cached triton-3.6.0-cp310-cp310-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (188.1 MB)
Using cached torchvision-0.26.0-cp310-cp310-manylinux_2_28_x86_64.whl (7.5 MB)
  Attempting uninstall: triton
    Found existing installation: triton 2.2.0
    Uninstalling triton-2.2.0:
      Successfully uninstalled triton-2.2.0
  Attempting uninstall: torch━━━━━━━━━━━━━━━━━━━ 0/3 [triton]
    Found existing installation: torch 2.2.2+cu1210/3 [triton]
    Uninstalling torch-2.2.2+cu121:━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [torch]
      Successfully uninstalled torch-2.2.2+cu121━━━━━━━━━━━━━━━━━━ 1/3 [torch]
  Attempting uninstall: torchvision━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
denoiser 0.1.5 requires torchaudio>=0.5, which is not installed.
denoiser 0.1.5 requires hydra-core<1.0, but you have hydra-core 1.3.2 which is incompatible.


0

In [4]:
pip uninstall -y torch torchvision torchaudio

Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Successfully uninstalled torch-2.11.0
Found existing installation: torchvision 0.26.0
Uninstalling torchvision-0.26.0:
  Successfully uninstalled torchvision-0.26.0
Note: you may need to restart the kernel to use updated packages.


In [5]:
!pip install torch==2.2.2+cu121 torchvision==0.17.2+cu121 --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.2.2%2Bcu121-cp310-cp310-linux_x86_64.whl (757.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.17.2%2Bcu121-cp310-cp310-linux_x86_64.whl (7.0 MB)
  Using cached https://download-r2.pytorch.org/whl/triton-2.2.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (167.9 MB)
  Attempting uninstall: triton
    Found existing installation: triton 3.6.0
    Uninstalling triton-3.6.0:
      Successfully uninstalled triton-3.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torchvision] [torchvision]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
denoiser 0.1.5 requires torchaudio>=0.5, which is not installed.
denoiser 0.1.5 requires hydra-core<1.0, but you have hydra-core 1.3.2 which is incompatible.


In [6]:
%pip uninstall -y torch torchvision torchaudio

Found existing installation: torch 2.2.2+cu121
Uninstalling torch-2.2.2+cu121:
  Successfully uninstalled torch-2.2.2+cu121
Found existing installation: torchvision 0.17.2+cu121
Uninstalling torchvision-0.17.2+cu121:
  Successfully uninstalled torchvision-0.17.2+cu121
Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install --no-cache-dir torch==2.2.2+cu121 torchvision==0.17.2+cu121 \
--index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.1/757.3 MB 70.3 MB/s  0:00:10m
  Resuming download https://download-r2.pytorch.org/whl/cu121/torch-2.2.2%2Bcu121-cp310-cp310-linux_x86_64.whl (113.1 MB/757.3 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.3/757.3 MB 116.3 MB/s  0:00:0500:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 118.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torchvision] [torchvision]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
denoiser 0.1.5 requires torchaudio>=0.5, which is not installed.
denoiser 0.1.5 requires hydra-core<1.0, but you have hydra-core 1.3.2 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install numpy==1.26.4

Note: you may need to restart the kernel to use updated packages.


In [9]:
import torch, torchvision
print(torch.__version__)
print(torchvision.__version__)

/home/teaching/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2+cu121
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
2026-04-29 08:58:29.394697564 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


2.2.2+cu121
0.17.2+cu121


In [10]:
import os
import copy
import time
import json
import random
import numpy as np
import matplotlib
matplotlib.use('Agg')          # non-interactive backend — safe on remote GPUs
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from torchvision.models import Inception_V3_Weights

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    print(f'VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

Using device : cuda
GPU          : NVIDIA RTX A5000
VRAM         : 25.42 GB


## 2. Configuration

In [11]:
# ══════════════════════════════════════════════════════════════════════════════
#  SET THIS to the path of your dataset on the remote GPU server
#  The folder must contain:  real/   and   attack/   subfolders
# ══════════════════════════════════════════════════════════════════════════════
DATA_DIR   = '/home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/DL_ProjectP12_clean'   # e.g. '/data/P12/DL_ProjectP12_clean'
OUTPUT_DIR = '/home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/output_inceptionv3'    # all results saved here
# ══════════════════════════════════════════════════════════════════════════════

# Create output sub-folders
DIRS = {
    'root'    : OUTPUT_DIR,
    'models'  : os.path.join(OUTPUT_DIR, 'models'),
    'plots'   : os.path.join(OUTPUT_DIR, 'plots'),
    'metrics' : os.path.join(OUTPUT_DIR, 'metrics'),
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

CFG = dict(
    data_dir      = DATA_DIR,
    image_size    = 299,
    batch_size    = 32,
    num_classes   = 2,
    num_workers   = 4,
    max_epochs    = 120,
    patience      = 10,
    lr            = 1e-4,
    lr_factor     = 0.5,
    lr_patience   = 5,
    weight_decay  = 1e-4,
    train_frac    = 0.80,
    val_frac      = 0.10,   # test = remaining 10%
    classes       = ['attack', 'real'],  # alphabetical = ImageFolder default
    dirs          = DIRS,
)

print('Output directory structure:')
for k, v in DIRS.items():
    print(f'  {k:8s} -> {v}')
print(f'\nDataset root : {os.path.abspath(DATA_DIR)}')

Output directory structure:
  root     -> /home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/output_inceptionv3
  models   -> /home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/output_inceptionv3/models
  plots    -> /home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/output_inceptionv3/plots
  metrics  -> /home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/output_inceptionv3/metrics

Dataset root : /home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/DL_ProjectP12_clean


## 3. Dataset Verification

In [12]:
from pathlib import Path

data_path = Path(CFG['data_dir'])
assert data_path.exists(), f'DATA_DIR not found: {data_path.resolve()}'

valid_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp', '.pgm'}

total = 0
print('Class distribution:')

for cls in CFG['classes']:
    cls_path = data_path / cls
    assert cls_path.exists(), f'Missing class folder: {cls_path}'
    
    # 🔥 recursive search (fix)
    images = [f for f in cls_path.rglob('*') if f.suffix.lower() in valid_ext]
    
    n = len(images)
    total += n
    
    print(f'  {cls:>12s}/  ->  {n:>5d} images')

print(f'  {"TOTAL":>12s}      {total:>5d} images')
print('\nFolder structure OK ✓')

Class distribution:
        attack/  ->   5900 images
          real/  ->   5900 images
         TOTAL      11800 images

Folder structure OK ✓


## 4. Transforms

In [13]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]
SZ   = CFG['image_size']

# Shared evaluation transform (val + test, both pipelines)
eval_tf = transforms.Compose([
    transforms.Resize((SZ, SZ)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Pipeline A: no augmentation
train_tf_no_aug = transforms.Compose([
    transforms.Resize((SZ, SZ)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Pipeline B: with augmentation
train_tf_aug = transforms.Compose([
    transforms.Resize((SZ, SZ)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

print('Transforms ready:')
print(f'  No-Aug  train : Resize({SZ}) -> ToTensor -> Normalize')
print(f'  Aug     train : Resize({SZ}) -> HFlip(p=0.5) -> Rotation(+-20 deg) -> ToTensor -> Normalize')
print(f'  Eval (shared) : Resize({SZ}) -> ToTensor -> Normalize')

Transforms ready:
  No-Aug  train : Resize(299) -> ToTensor -> Normalize
  Aug     train : Resize(299) -> HFlip(p=0.5) -> Rotation(+-20 deg) -> ToTensor -> Normalize
  Eval (shared) : Resize(299) -> ToTensor -> Normalize


## 5. Build DataLoaders (80 / 10 / 10 stratified split)

In [14]:
from collections import Counter
from torchvision import datasets

ds = datasets.ImageFolder(CFG['data_dir'])
print("Total samples:", len(ds))
print("Class distribution:", Counter(ds.targets))

Total samples: 11807
Class distribution: Counter({0: 5900, 3: 5900, 2: 6, 1: 1})


In [15]:
import numpy as np
from pathlib import Path
from PIL import Image
from collections import Counter
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader, Subset

# ─────────────────────────────────────────────────────────────
# ✅ Custom Dataset (handles nested folders correctly)
# ─────────────────────────────────────────────────────────────
class PADDataset(Dataset):
    def __init__(self, root, transform=None):
        self.samples = []
        self.transform = transform

        root = Path(root)

        for label_name, label in [('attack', 0), ('real', 1)]:
            class_dir = root / label_name

            for img_path in class_dir.rglob('*'):
                if img_path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
                    self.samples.append((img_path, label))

        if len(self.samples) == 0:
            raise RuntimeError("No images found. Check dataset path.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img, label


# ─────────────────────────────────────────────────────────────
# ✅ Loader Function (FIXED)
# ─────────────────────────────────────────────────────────────
def make_loaders(data_dir, train_tf, eval_tf, cfg, seed=42):

    base_ds = PADDataset(data_dir)

    # Extract correct binary targets
    targets = np.array([label for _, label in base_ds.samples])
    indices = np.arange(len(base_ds))

    print("Class distribution:", Counter(targets))

    # ── Stratified Split ──────────────────────────────────────
    idx_tr, idx_tmp, y_tr, y_tmp = train_test_split(
        indices, targets,
        test_size=1.0 - cfg['train_frac'],
        stratify=targets,
        random_state=seed
    )

    idx_val, idx_te = train_test_split(
        idx_tmp,
        test_size=0.5,
        stratify=y_tmp,
        random_state=seed
    )

    # ── Datasets with transforms ──────────────────────────────
    ds_tr  = PADDataset(data_dir, transform=train_tf)
    ds_val = PADDataset(data_dir, transform=eval_tf)
    ds_te  = PADDataset(data_dir, transform=eval_tf)

    # ── DataLoaders ───────────────────────────────────────────
    kw = dict(num_workers=cfg['num_workers'], pin_memory=True)

    loaders = {
        'train': DataLoader(Subset(ds_tr, idx_tr),
                            batch_size=cfg['batch_size'], shuffle=True, **kw),

        'val': DataLoader(Subset(ds_val, idx_val),
                          batch_size=cfg['batch_size'], shuffle=False, **kw),

        'test': DataLoader(Subset(ds_te, idx_te),
                           batch_size=cfg['batch_size'], shuffle=False, **kw),
    }

    print(f"train={len(idx_tr)}  val={len(idx_val)}  test={len(idx_te)}")

    class_to_idx = {'attack': 0, 'real': 1}
    return loaders, class_to_idx


# ─────────────────────────────────────────────────────────────
# ✅ RUN PIPELINES
# ─────────────────────────────────────────────────────────────

print('-- Pipeline A: No Augmentation --')
loaders_no, class_to_idx = make_loaders(
    CFG['data_dir'], train_tf_no_aug, eval_tf, CFG
)

print('\n-- Pipeline B: With Augmentation --')
loaders_aug, _ = make_loaders(
    CFG['data_dir'], train_tf_aug, eval_tf, CFG
)

idx_to_class = {v: k for k, v in class_to_idx.items()}
print('\nLabel map:', idx_to_class)

-- Pipeline A: No Augmentation --
Class distribution: Counter({0: 5900, 1: 5900})
train=9440  val=1180  test=1180

-- Pipeline B: With Augmentation --
Class distribution: Counter({0: 5900, 1: 5900})
train=9440  val=1180  test=1180

Label map: {0: 'attack', 1: 'real'}


## 6. Model Builder — Pretrained InceptionV3

In [16]:
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--force-reinstall",
    "torch==2.2.2",
    "torchvision==0.17.2"
])

  Using cached torch-2.2.2-cp310-cp310-manylinux1_x86_64.whl.metadata (26 kB)
  Using cached torchvision-0.17.2-cp310-cp310-manylinux1_x86_64.whl.metadata (6.6 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylin

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
denoiser 0.1.5 requires torchaudio>=0.5, which is not installed.
dm-tree 0.1.9 requires absl-py>=0.6.1, which is not installed.
dm-tree 0.1.9 requires wrapt>=1.11.2, which is not installed.
google-genai 1.73.1 requires distro<2,>=1.7.0, which is not installed.
kapre 0.3.7 requires tensorflow>=2.0.0, which is not installed.
librosa 0.11.0 requires soundfile>=0.12.1, which is not installed.
nbconvert 6.5.0 requires defusedxml, which is not installed.
sacrebleu 2.5.1 requires colorama, which is not installed.
tensorflow-datasets 4.9.9 requires absl-py, which is not installed.
tensorflow-datasets 4.9.9 requires wrapt, which is not installed.
datasets 3.5.1 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.3.0 which is incompatible.
datasets 3.5.1 requires requests>=2.32.2, but you have requests 2.31

0

In [17]:
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "uninstall", "-y",
    "torch", "torchvision", "torchaudio"
])

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "torch==2.2.2",
    "torchvision==0.17.2",
    "torchaudio==2.2.2"
])

Found existing installation: torch 2.2.2
Uninstalling torch-2.2.2:
  Successfully uninstalled torch-2.2.2
Found existing installation: torchvision 0.17.2
Uninstalling torchvision-0.17.2:
  Successfully uninstalled torchvision-0.17.2


  Using cached torch-2.2.2-cp310-cp310-manylinux1_x86_64.whl.metadata (26 kB)
  Using cached torchvision-0.17.2-cp310-cp310-manylinux1_x86_64.whl.metadata (6.6 kB)
  Using cached torchaudio-2.2.2-cp310-cp310-manylinux1_x86_64.whl.metadata (6.4 kB)
Using cached torch-2.2.2-cp310-cp310-manylinux1_x86_64.whl (755.5 MB)
Using cached torchvision-0.17.2-cp310-cp310-manylinux1_x86_64.whl (6.9 MB)
Using cached torchaudio-2.2.2-cp310-cp310-manylinux1_x86_64.whl (3.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torchaudio]3 [torchaudio]]


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
denoiser 0.1.5 requires hydra-core<1.0, but you have hydra-core 1.3.2 which is incompatible.


0

In [18]:
import torch, torchvision
print(torch.__version__)
print(torchvision.__version__)

2.2.2+cu121
0.17.2+cu121


In [19]:
CFG = {
    'num_classes': 2,
    'batch_size': 4,        # or 6 as you planned
    'num_workers': 2,
    'train_frac': 0.8,
    'data_dir': '/home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/DL_ProjectP12_clean'
}

In [20]:
import torch.nn as nn
from torchvision import models
from torchvision.models import Inception_V3_Weights

In [21]:
def build_inception_v3(num_classes=2):
    """
    Load pretrained InceptionV3 and replace BOTH classifiers:
      model.fc             (main head,  in_features=2048)
      model.AuxLogits.fc   (aux  head,  in_features=768)
    """
    model = models.inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, num_classes)
    return model


# Sanity check
_tmp = build_inception_v3(CFG['num_classes'])
print(f'Main head        : {_tmp.fc}')
print(f'Aux  head        : {_tmp.AuxLogits.fc}')
total_p = sum(p.numel() for p in _tmp.parameters())
train_p = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f'Total params     : {total_p:,}')
print(f'Trainable params : {train_p:,}')
del _tmp
print('Model builder OK ✓')

Main head        : Linear(in_features=2048, out_features=2, bias=True)
Aux  head        : Linear(in_features=768, out_features=2, bias=True)
Total params     : 24,348,900
Trainable params : 24,348,900
Model builder OK ✓


## 7. Training & Evaluation Utilities

In [22]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0

    for inputs, labels in tqdm(loader, leave=False, desc='  train'):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        outputs = model(inputs)   # (main, aux) tuple during training
        if isinstance(outputs, tuple):
            main_out, aux_out = outputs
            # Auxiliary loss weight = 0.4 (from original Inception paper)
            loss = criterion(main_out, labels) + 0.4 * criterion(aux_out, labels)
        else:
            main_out = outputs
            loss = criterion(main_out, labels)

        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * inputs.size(0)
        preds     = main_out.argmax(dim=1)
        correct  += (preds == labels).sum().item()
        total    += labels.size(0)

    return loss_sum / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    all_preds, all_labels    = [], []

    for inputs, labels in tqdm(loader, leave=False, desc='  eval '):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        out = model(inputs)
        if isinstance(out, tuple):
            out = out[0]           # only main head in eval mode
        loss  = criterion(out, labels)
        preds = out.argmax(dim=1)

        loss_sum += loss.item() * inputs.size(0)
        correct  += (preds == labels).sum().item()
        total    += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return (loss_sum / total, correct / total,
            np.array(all_preds), np.array(all_labels))


print('Utilities defined ✓')

Utilities defined ✓


## 8. Main Training Function

In [23]:
def run_pipeline(name, loaders, cfg):
    """
    Full training loop:
      Adam + ReduceLROnPlateau + early stopping (patience=10) + best-model checkpoint.
    Best model  ->  output_inceptionv3/models/best_{name}.pth
    History     ->  output_inceptionv3/metrics/history_{name}.json
    """
    print(f'\n{"="*70}')
    print(f'  PIPELINE : {name}')
    print(f'{"="*70}')

    model     = build_inception_v3(cfg['num_classes']).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(),
                           lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min',
        factor=cfg['lr_factor'], patience=cfg['lr_patience'], verbose=True)

    history = {'train_loss': [], 'train_acc': [],
               'val_loss':   [], 'val_acc':   []}

    best_val_loss     = float('inf')
    best_weights      = None
    epochs_no_improve = 0
    ckpt_path         = os.path.join(cfg['dirs']['models'], f'best_{name}.pth')

    for epoch in range(1, cfg['max_epochs'] + 1):
        t0 = time.time()

        tr_loss, tr_acc = train_one_epoch(
            model, loaders['train'], criterion, optimizer)
        vl_loss, vl_acc, _, _ = evaluate(
            model, loaders['val'], criterion)

        scheduler.step(vl_loss)

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)

        lr_now  = optimizer.param_groups[0]['lr']
        elapsed = time.time() - t0
        tag     = ''

        if vl_loss < best_val_loss:
            best_val_loss     = vl_loss
            best_weights      = copy.deepcopy(model.state_dict())
            torch.save(best_weights, ckpt_path)
            epochs_no_improve = 0
            tag               = '  <- best'
        else:
            epochs_no_improve += 1

        print(f'Ep {epoch:03d}/{cfg["max_epochs"]}  '
              f'tr_loss={tr_loss:.4f}  tr_acc={tr_acc:.4f}  '
              f'vl_loss={vl_loss:.4f}  vl_acc={vl_acc:.4f}  '
              f'lr={lr_now:.1e}  {elapsed:.1f}s{tag}')

        if epochs_no_improve >= cfg['patience']:
            print(f'\n  Early stopping at epoch {epoch} '
                  f'(no improvement for {cfg["patience"]} consecutive epochs).')
            break

    model.load_state_dict(best_weights)
    print(f'\nBest val loss : {best_val_loss:.4f}')
    print(f'Checkpoint    : {ckpt_path}')

    hist_path = os.path.join(cfg['dirs']['metrics'], f'history_{name}.json')
    with open(hist_path, 'w') as f:
        json.dump(history, f, indent=2)
    print(f'History       : {hist_path}')

    return model, history


print('run_pipeline() defined ✓')

run_pipeline() defined ✓


## 9. Train — Pipeline A: No Augmentation

In [24]:
CFG = {
    'num_classes': 2,
    'batch_size': 6,
    'num_workers': 2,
    'train_frac': 0.8,
    'data_dir': '/home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/DL_ProjectP12_clean',

    # 🔥 ADD THESE
    'lr': 1e-4,
    'epochs': 100,
    'weight_decay': 1e-4
}

In [25]:
from torchvision import transforms

train_tf_no_aug = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor()
])

train_tf_aug = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

eval_tf = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor()
])

In [26]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset
import numpy as np
from collections import Counter

def make_loaders(data_dir, train_tf, eval_tf, cfg, seed=42):

    base_ds = PADDataset(data_dir)

    targets = np.array([label for _, label in base_ds.samples])
    indices = np.arange(len(base_ds))

    print("Class distribution:", Counter(targets))

    idx_tr, idx_tmp, y_tr, y_tmp = train_test_split(
        indices, targets,
        test_size=1.0 - cfg['train_frac'],
        stratify=targets,
        random_state=seed
    )

    idx_val, idx_te = train_test_split(
        idx_tmp,
        test_size=0.5,
        stratify=y_tmp,
        random_state=seed
    )

    ds_tr  = PADDataset(data_dir, transform=train_tf)
    ds_val = PADDataset(data_dir, transform=eval_tf)
    ds_te  = PADDataset(data_dir, transform=eval_tf)

    kw = dict(num_workers=cfg['num_workers'], pin_memory=True)

    loaders = {
        'train': DataLoader(Subset(ds_tr, idx_tr),
                            batch_size=cfg['batch_size'], shuffle=True, **kw),
        'val': DataLoader(Subset(ds_val, idx_val),
                          batch_size=cfg['batch_size'], shuffle=False, **kw),
        'test': DataLoader(Subset(ds_te, idx_te),
                           batch_size=cfg['batch_size'], shuffle=False, **kw),
    }

    print(f"train={len(idx_tr)}  val={len(idx_val)}  test={len(idx_te)}")

    class_to_idx = {'attack': 0, 'real': 1}
    return loaders, class_to_idx

In [27]:
from torch.utils.data import Dataset
from pathlib import Path
from PIL import Image

class PADDataset(Dataset):
    def __init__(self, root, transform=None):
        self.samples = []
        self.transform = transform

        root = Path(root)

        for label_name, label in [('attack', 0), ('real', 1)]:
            for img_path in (root / label_name).rglob('*'):
                if img_path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
                    self.samples.append((img_path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')

        if self.transform:
            img = self.transform(img)

        return img, label

In [28]:
loaders_no, class_to_idx = make_loaders(
    CFG['data_dir'], train_tf_no_aug, eval_tf, CFG
)

print("Loaders created:", loaders_no.keys())

Class distribution: Counter({0: 5900, 1: 5900})
train=9440  val=1180  test=1180
Loaders created: dict_keys(['train', 'val', 'test'])


In [29]:
print('loaders_no' in globals())

True


In [30]:
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--force-reinstall",
    "torch==2.2.2+cu121",
    "torchvision==0.17.2+cu121",
    "--index-url", "https://download.pytorch.org/whl/cu121"
])

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.2.2%2Bcu121-cp310-cp310-linux_x86_64.whl (757.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.17.2%2Bcu121-cp310-cp310-linux_x86_64.whl (7.0 MB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached https://download.pytorch.org/whl/typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.4.2-py3-none-any.whl.metadata (6.3 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached https://download.pytorch.org/whl/cu121/nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_6

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dm-tree 0.1.9 requires absl-py>=0.6.1, which is not installed.
dm-tree 0.1.9 requires wrapt>=1.11.2, which is not installed.
google-genai 1.73.1 requires distro<2,>=1.7.0, which is not installed.
kapre 0.3.7 requires tensorflow>=2.0.0, which is not installed.
librosa 0.11.0 requires soundfile>=0.12.1, which is not installed.
nbconvert 6.5.0 requires defusedxml, which is not installed.
sacrebleu 2.5.1 requires colorama, which is not installed.
tensorflow-datasets 4.9.9 requires absl-py, which is not installed.
tensorflow-datasets 4.9.9 requires wrapt, which is not installed.
datasets 3.5.1 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
datasets 3.5.1 requires requests>=2.32.2, but you have requests 2.31.0 which is incompatible.
denoiser 0.1.5 requires hydra-core<1.0,

0

In [31]:
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("Using device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA RTX A5000


In [32]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

def run_pipeline(name, loaders, cfg):
    print("="*60)
    print(f"PIPELINE: {name}")
    print("="*60)

    model = build_inception_v3(cfg['num_classes']).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=cfg['lr'])

    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(cfg['epochs']):
        print(f"\nEpoch [{epoch+1}/{cfg['epochs']}]")

        # ───── TRAIN ─────
        model.train()
        running_loss = 0

        for imgs, labels in tqdm(loaders['train']):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()

            outputs, aux_outputs = model(imgs)  # Inception returns 2 outputs

            loss1 = criterion(outputs, labels)
            loss2 = criterion(aux_outputs, labels)

            loss = loss1 + 0.4 * loss2  # important

            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(loaders['train'])

        # ───── VALIDATION ─────
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for imgs, labels in loaders['val']:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

                outputs = model(imgs)  # only main output in eval
                if isinstance(outputs, tuple):
                    outputs = outputs[0]

                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, preds = torch.max(outputs, 1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_loss /= len(loaders['val'])
        val_acc = correct / total

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Loss  : {val_loss:.4f}")
        print(f"Val Acc   : {val_acc:.4f}")

    return model, history

In [33]:
%pip install numpy==1.26.4

  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dm-tree 0.1.9 requires absl-py>=0.6.1, which is not installed.
dm-tree 0.1.9 requires wrapt>=1.11.2, which is not installed.
kapre 0.3.7 requires tensorflow>=2.0.0, which is not installed.
librosa 0.11.0 requires soundfile>=0.12.1, which is not installed.
sacrebleu 2.5.1 requires colorama, which is not installed.
tensorflow-datasets 4.9.9 requires absl-py, which is not installed.
tensorflow-datasets 4.9.9 requires wrapt, which is not installed.
datasets 3.5.1 requires fsspe

In [34]:
import numpy as np
print(np.__version__)

1.26.4


In [35]:
model_no_aug, history_no_aug = run_pipeline('NoAug', loaders_no, CFG)

PIPELINE: NoAug

Epoch [1/100]


100%|██████████| 1574/1574 [00:59<00:00, 26.41it/s]


Train Loss: 0.2432
Val Loss  : 0.2168
Val Acc   : 0.9280

Epoch [2/100]


100%|██████████| 1574/1574 [01:04<00:00, 24.50it/s]


Train Loss: 0.1037
Val Loss  : 0.0880
Val Acc   : 0.9763

Epoch [3/100]


...

Train Loss: 0.0041
Val Loss  : 0.0112
Val Acc   : 0.9975

Epoch [99/100]


100%|██████████| 1574/1574 [01:12<00:00, 21.74it/s]


Train Loss: 0.0051
Val Loss  : 0.0407
Val Acc   : 0.9941

Epoch [100/100]


100%|██████████| 1574/1574 [01:12<00:00, 21.74it/s]


Train Loss: 0.0103
Val Loss  : 0.1891
Val Acc   : 0.9814


## 10. Train — Pipeline B: With Augmentation

In [36]:
model_aug, history_aug = run_pipeline('WithAug', loaders_aug, CFG)

PIPELINE: WithAug

Epoch [1/100]


100%|██████████| 295/295 [00:36<00:00,  8.19it/s]


Train Loss: 0.1515
Val Loss  : 0.0208
Val Acc   : 0.9949

Epoch [2/100]


100%|██████████| 295/295 [00:36<00:00,  8.15it/s]


Train Loss: 0.0573
Val Loss  : 0.0107
Val Acc   : 0.9949

Epoch [3/100]


...

Train Loss: 0.0012
Val Loss  : 0.0099
Val Acc   : 0.9975

Epoch [99/100]


100%|██████████| 295/295 [00:34<00:00,  8.56it/s]


Train Loss: 0.0004
Val Loss  : 0.0074
Val Acc   : 0.9992

Epoch [100/100]


100%|██████████| 295/295 [00:34<00:00,  8.57it/s]


Train Loss: 0.0002
Val Loss  : 0.0063
Val Acc   : 0.9992


## 11. Compute Test-Set Metrics

In [39]:
CFG['dirs'] = {
    'metrics': '/home/teaching/DL-12/DL_12PROJECT_VULNERABILITY/output_inceptionv3/metrics'
}

In [41]:
CFG['dirs'] = {
    'metrics': './metrics'
}

import os
os.makedirs(CFG['dirs']['metrics'], exist_ok=True)

In [43]:
import os, json
from sklearn.metrics import *

def compute_metrics(y_true, y_pred):

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)

    cm = confusion_matrix(y_true, y_pred)

    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
    else:
        tn = fp = fn = tp = 0

    apcer = fp / (tn + fp) if (tn + fp) > 0 else 0.0
    bpcer = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    acer  = (apcer + bpcer) / 2.0

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "apcer": apcer,
        "bpcer": bpcer,
        "acer": acer,
        "confusion_matrix": cm.tolist()
    }


# Ensure folder exists
os.makedirs(CFG['dirs']['metrics'], exist_ok=True)


# Evaluate
_, _, preds_no,  labels_no  = evaluate(model_no_aug, loaders_no['test'],  criterion)
_, _, preds_aug, labels_aug = evaluate(model_aug,    loaders_aug['test'], criterion)


# Compute
metrics_no  = compute_metrics(labels_no,  preds_no)
metrics_aug = compute_metrics(labels_aug, preds_aug)


# Save
for tag, m in [('NoAug', metrics_no), ('WithAug', metrics_aug)]:
    path = os.path.join(CFG['dirs']['metrics'], f'test_metrics_{tag}.json')
    with open(path, 'w') as f:
        json.dump(m, f, indent=2)
    print(f'Saved: {path}')


# Print
for tag, m, y_t, y_p in [
    ('NoAug',   metrics_no,  labels_no,  preds_no),
    ('WithAug', metrics_aug, labels_aug, preds_aug)
]:
    print(f'\n-- {tag} --------------------------')
    print(f'Accuracy : {m["accuracy"]:.4f}')
    print(f'Precision: {m["precision"]:.4f}')
    print(f'Recall   : {m["recall"]:.4f}')
    print(f'F1       : {m["f1"]:.4f}')
    print(f'APCER    : {m["apcer"]:.4f}')
    print(f'BPCER    : {m["bpcer"]:.4f}')
    print(f'ACER     : {m["acer"]:.4f}')

    # FIX: define class names directly
class_names = ['attack', 'real']
print(classification_report(y_t, y_p, target_names=class_names))

Saved: ./metrics/test_metrics_NoAug.json
Saved: ./metrics/test_metrics_WithAug.json

-- NoAug --------------------------
Accuracy : 0.9898
Precision: 1.0000
Recall   : 0.9797
F1       : 0.9897
APCER    : 0.0000
BPCER    : 0.0203
ACER     : 0.0102

-- WithAug --------------------------
Accuracy : 1.0000
Precision: 1.0000
Recall   : 1.0000
F1       : 1.0000
APCER    : 0.0000
BPCER    : 0.0000
ACER     : 0.0000
              precision    recall  f1-score   support

      attack       1.00      1.00      1.00       590
        real       1.00      1.00      1.00       590

    accuracy                           1.00      1180
   macro avg       1.00      1.00      1.00      1180
weighted avg       1.00      1.00      1.00      1180



## 12. Save All Plots to output_inceptionv3/plots/

In [46]:
import os

CFG['dirs'] = {
    'metrics': './metrics',
    'plots': './plots'
}

# create folders
for p in CFG['dirs'].values():
    os.makedirs(p, exist_ok=True)

In [47]:
def savefig(filename):
    path = os.path.join(CFG['dirs']['plots'], filename)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Saved: {path}')


# 12-A  Training curves (loss & accuracy) per pipeline
def plot_curves(history, title, fname):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='royalblue')
    axes[0].plot(epochs, history['val_loss'],   label='Val Loss',   color='tomato')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # axes[1].plot(epochs, history['train_acc'], label='Train Acc', color='royalblue')
    axes[1].plot(epochs, history['val_acc'],   label='Val Acc',   color='tomato')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    savefig(fname)


plot_curves(history_no_aug, 'InceptionV3 - No Augmentation',   'curves_NoAug.png')
plot_curves(history_aug,    'InceptionV3 - With Augmentation', 'curves_WithAug.png')

Saved: ./plots/curves_NoAug.png
Saved: ./plots/curves_WithAug.png


In [49]:
# Define class names explicitly (based on your dataset)
class_names = ['attack', 'real']

In [51]:
# 12-B  Confusion matrices
def plot_confusion(y_true, y_pred, label_names, title, fname):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names,
                linewidths=0.5, ax=ax)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual',    fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    savefig(fname)
plot_confusion(labels_no,  preds_no,  class_names,
               'Confusion Matrix - No Augmentation',   'cm_NoAug.png')

plot_confusion(labels_aug, preds_aug, class_names,
               'Confusion Matrix - With Augmentation', 'cm_WithAug.png')

Saved: ./plots/cm_NoAug.png
Saved: ./plots/cm_WithAug.png


In [52]:
# 12-C  Overlaid validation curves — both pipelines on one chart
def plot_overlay(h_no, h_aug, fname='overlay_val_curves.png'):
    e_no  = range(1, len(h_no['val_loss'])  + 1)
    e_aug = range(1, len(h_aug['val_loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Validation Curves — No Aug vs With Aug',
                 fontsize=14, fontweight='bold')

    axes[0].plot(e_no,  h_no['val_loss'],  label='No Aug',   color='royalblue',  lw=2)
    axes[0].plot(e_aug, h_aug['val_loss'], label='With Aug', color='darkorange', lw=2)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val Loss')
    axes[0].set_title('Validation Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].plot(e_no,  h_no['val_acc'],  label='No Aug',   color='royalblue',  lw=2)
    axes[1].plot(e_aug, h_aug['val_acc'], label='With Aug', color='darkorange', lw=2)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val Accuracy')
    axes[1].set_title('Validation Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    savefig(fname)


plot_overlay(history_no_aug, history_aug)

Saved: ./plots/overlay_val_curves.png


In [53]:
# 12-D  Bar chart — all metrics side-by-side
def plot_bar_comparison(m_no, m_aug, fname='bar_comparison.png'):
    keys   = ['accuracy', 'precision', 'recall', 'f1', 'apcer', 'bpcer', 'acer']
    labels = ['Accuracy', 'Precision', 'Recall', 'F1', 'APCER', 'BPCER', 'ACER']
    v_no   = [m_no[k]  for k in keys]
    v_aug  = [m_aug[k] for k in keys]

    x, w = np.arange(len(keys)), 0.35
    fig, ax = plt.subplots(figsize=(13, 6))
    b1 = ax.bar(x - w/2, v_no,  w, label='No Aug',   color='royalblue',  alpha=0.85)
    b2 = ax.bar(x + w/2, v_aug, w, label='With Aug', color='darkorange', alpha=0.85)

    for bar in [*b1, *b2]:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.008,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylim(0, 1.13); ax.set_ylabel('Score', fontsize=12)
    ax.set_title('InceptionV3 Test Metrics — No Aug vs With Aug',
                 fontsize=13, fontweight='bold')
    ax.legend(fontsize=11); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    savefig(fname)


plot_bar_comparison(metrics_no, metrics_aug)

Saved: ./plots/bar_comparison.png


## 13. Final Comparison Table

In [54]:
def print_and_save_comparison(m_no, m_aug, cfg):
    keys   = ['accuracy', 'precision', 'recall', 'f1', 'apcer', 'bpcer', 'acer']
    labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'APCER', 'BPCER', 'ACER']
    # error metrics: lower is better
    lower_is_better = {'apcer', 'bpcer', 'acer'}

    sep = '=' * 72
    print(f'\n{sep}')
    print('  InceptionV3 — Final Test-Set Comparison  (P12 PAD Project)')
    print(sep)
    hdr = f'{"Metric":<14}  {"No Aug":>10}  {"With Aug":>10}  {"Delta":>10}  {"Winner":>10}'
    print(hdr)
    print('-' * 72)

    summary = {}
    for k, lbl in zip(keys, labels):
        vn    = m_no[k]
        va    = m_aug[k]
        delta = va - vn
        if k in lower_is_better:
            winner = 'With Aug' if delta < 0 else ('No Aug' if delta > 0 else 'Tie')
        else:
            winner = 'With Aug' if delta > 0 else ('No Aug' if delta < 0 else 'Tie')
        print(f'{lbl:<14}  {vn:>10.4f}  {va:>10.4f}  {delta:>+10.4f}  {winner:>10}')
        summary[k] = dict(no_aug=vn, with_aug=va, delta=delta, winner=winner)

    print(sep)

    path = os.path.join(cfg['dirs']['metrics'], 'final_comparison.json')
    with open(path, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f'\nComparison saved: {path}')


print_and_save_comparison(metrics_no, metrics_aug, CFG)


  InceptionV3 — Final Test-Set Comparison  (P12 PAD Project)
Metric              No Aug    With Aug       Delta      Winner
------------------------------------------------------------------------
Accuracy            0.9898      1.0000     +0.0102    With Aug
Precision           1.0000      1.0000     +0.0000         Tie
Recall              0.9797      1.0000     +0.0203    With Aug
F1-Score            0.9897      1.0000     +0.0103    With Aug
APCER               0.0000      0.0000     +0.0000         Tie
BPCER               0.0203      0.0000     -0.0203    With Aug
ACER                0.0102      0.0000     -0.0102    With Aug

Comparison saved: ./metrics/final_comparison.json


## 14. List All Saved Output Files

In [55]:
print(f'\noutput_inceptionv3/')
for root, dirs, files in os.walk(OUTPUT_DIR):
    dirs.sort()
    level     = root.replace(OUTPUT_DIR, '').count(os.sep)
    indent    = '    ' * level
    sub_ind   = '    ' * (level + 1)
    if level > 0:
        print(f'{indent}{os.path.basename(root)}/')
    for fname in sorted(files):
        fpath = os.path.join(root, fname)
        size  = os.path.getsize(fpath)
        if size >= 1e6:
            sz_str = f'{size/1e6:.1f} MB'
        else:
            sz_str = f'{size/1e3:.1f} KB'
        print(f'{sub_ind}{fname:<50s}  {sz_str}')


output_inceptionv3/
    metrics/
        test_metrics_NoAug.json                             0.3 KB
        test_metrics_WithAug.json                           0.2 KB
    models/
    plots/


---
## Summary

All outputs are saved under `output_inceptionv3/`:

| Sub-folder | Files |
|---|---|
| `models/` | `best_NoAug.pth` · `best_WithAug.pth` |
| `plots/` | `curves_NoAug.png` · `curves_WithAug.png` · `cm_NoAug.png` · `cm_WithAug.png` · `overlay_val_curves.png` · `bar_comparison.png` |
| `metrics/` | `history_NoAug.json` · `history_WithAug.json` · `test_metrics_NoAug.json` · `test_metrics_WithAug.json` · `final_comparison.json` |